# Notebook for FOV Visualization

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, FuncFormatter
import warnings
warnings.filterwarnings('ignore')

# Add root to path for utils
sys.path.append(os.path.abspath('../../'))

USE_FUNCTIONAL_MARKERS= False

CELLTYPE_MARKER_THRESHOLDS = {
    'Epithelial': {'Ki67': 2.5, 'HLADRDPDQ': 1.75},
    'CD8T': {'Ki67': 2.5, 'CD103': 2.0, 'CD69': 2.0, 'GZMB': 2.5},
    'CD4T': {'Ki67': 1.0, 'CD103': 1.0, 'CD69': 2.0, 'GZMB': 0.5},
    'Neutrophil': {'CD103': 2.75},
    'Neutrophil_CD15': {'CD103': 2.75}
}


In [ ]:
def _add_functional_subtypes(df):
    if not USE_FUNCTIONAL_MARKERS:
        df["functional_subtypes"] = [[] for _ in range(len(df))]
        return df
    functional_subtypes_list = [[] for _ in range(len(df))]
    idx_to_pos = {idx: i for i, idx in enumerate(df.index)}
    for base_type, markers in CELLTYPE_MARKER_THRESHOLDS.items():
        type_mask = df["cell type"] == base_type
        for marker, threshold in markers.items():
            if marker in df.columns:
                mask = type_mask & (df[marker] > threshold)
                subtype_label = f"{base_type}_{marker}+"
                matching_indices = df.index[mask]
                for idx in matching_indices:
                    functional_subtypes_list[idx_to_pos[idx]].append(subtype_label)
    df["functional_subtypes"] = functional_subtypes_list
    return df

def get_organ(row):
    if pd.notna(row.get("Localization")): return row["Localization"]
    cohort = str(row.get("Cohort", ""))
    if "Colon" in cohort: return "Colon"
    if "Duodenum" in cohort: return "Duodenum"
    return "Unknown"

print("Loading datasets...")
def get_data_dir():
    # Find the data directory by walking up from the current directory
    current_dir = os.path.abspath(os.getcwd())
    while current_dir != os.path.dirname(current_dir):
        potential_data = os.path.join(current_dir, 'data', 'MIBIGutCsv')
        if os.path.exists(potential_data):
            return potential_data
        current_dir = os.path.dirname(current_dir)
    return '../../data/MIBIGutCsv'  # Fallback

data_dir = get_data_dir()
cell_table_path = os.path.join(data_dir, 'cell_table.csv')
fov_meta_path = os.path.join(data_dir, 'fovs_metadata.csv')
biopsy_meta_path = os.path.join(data_dir, 'biopsy_metadata.csv')

df_cells = pd.read_csv(cell_table_path)
df_fovs = pd.read_csv(fov_meta_path)
df_biopsy = pd.read_csv(biopsy_meta_path)

# Merge FOV with Biopsy to get score columns and Localization
df_fovs = pd.merge(
    df_fovs,
    df_biopsy[['Biopsy_ID', 'Pathological score', 'Clinical score', 'Localization']],
    left_on='Patient',
    right_on='Biopsy_ID',
    how='left'
)
df_fovs["Organ"] = df_fovs.apply(get_organ, axis=1)

# Normalize Coordinates
fov_to_size = df_fovs.set_index('FOV')['Size [um]'].to_dict()
def normalize_coords(row):
    size = fov_to_size.get(row['fov'], 800)
    res = 1024 if size == 400 else 2048
    return row['centroid_x'] * (size / res), row['centroid_y'] * (size / res)

print("Normalizing coordinates...")
df_cells[['x_um', 'y_um']] = df_cells.apply(lambda row: pd.Series(normalize_coords(row)), axis=1)

print("Adding functional subtypes...")
df_cells = _add_functional_subtypes(df_cells)

print(f"Total Cells Loaded: {len(df_cells)}")


In [ ]:
# Generate Global Colormap
all_types = sorted(df_cells["cell type"].dropna().astype(str).unique())
try:
    palette = plt.colormaps.get_cmap("tab20b").resampled(max(len(all_types), 1))
except AttributeError:
    palette = plt.cm.get_cmap("tab20b", max(len(all_types), 1))
CELL_COLOR_MAP = {ct: palette(i) for i, ct in enumerate(all_types)}
OTHER_COLOR = (0.5, 0.5, 0.5)

def add_scale_bar_50um(ax, x_max, y_max):
    # Data is already in um, so 50um is just 50 units.
    bar_um = 50.0
    x_end = x_max - 25
    x_start = x_end - bar_um
    y_line = y_max - 25
    ax.plot([x_start, x_end], [y_line, y_line], color="black", linewidth=4, solid_capstyle="butt")
    ax.text((x_start + x_end) / 2, y_line - 12, "50 µm", color="black", ha="center", va="bottom", fontsize=10)


def plot_fov(fov_id, df_cells_all, df_fovs_all, target_cells=None):
    """
    Plots an FOV.
    - If target_cells is None: Plots all cells in the FOV normally.
    - If target_cells is provided: Grays out background cells and highlights targets using their original colors.
    """
    df_fov = df_cells_all[df_cells_all["fov"] == fov_id].copy()
    if df_fov.empty:
        print(f"No cells found for FOV {fov_id}")
        return
    
    size_um = df_fovs_all[df_fovs_all['FOV'] == fov_id]['Size [um]'].iloc[0] if not df_fovs_all[df_fovs_all['FOV'] == fov_id].empty else 400
    
    if size_um == 400:
        cell_size = 100
    else:
        cell_size = 50

    fig, ax = plt.subplots(figsize=(10, 10), facecolor="#f2f2f2")
    ax.set_facecolor("#eaeaeaff")
    
    # --- Conditional Plotting Logic ---
    if target_cells:
        title = f"Highlighted Rule in {fov_id}\n{', '.join(target_cells)}"
        
        # Plot 'Other' cells first (background)
        other_mask = ~df_fov['cell type'].isin(target_cells)
        if other_mask.any():
            g = df_fov[other_mask]
            ax.scatter(g["x_um"], g["y_um"], s=cell_size, c=[OTHER_COLOR], alpha=0.3, linewidths=0, label='Other')
            
        # Plot target cells grouped by type on top
        target_mask = df_fov['cell type'].isin(target_cells)
        if target_mask.any():
            for ct, g in df_fov[target_mask].groupby("cell type"):
                color = CELL_COLOR_MAP.get(ct, (0, 0, 0))
                # Plot halo
                # ax.scatter(g["x_um"], g["y_um"], s=40, c=[color], alpha=0.2, linewidths=0)
                # Plot center
                ax.scatter(g["x_um"], g["y_um"], s=cell_size, c=[color], label=ct, alpha=0.9, linewidths=0)
                
        # Legend only for the highlighted cells
        handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=CELL_COLOR_MAP.get(ct, 'black'), markeredgecolor="none", markersize=8, label=ct) for ct in target_cells]
        
    else:
        title = f"Full FOV: {fov_id}"
        # Plot all cells normally
        for ct, g in df_fov.groupby("cell type"):
            color = CELL_COLOR_MAP.get(ct, (1, 1, 1))
            # ax.scatter(g["x_um"], g["y_um"], s=40, c=[color], alpha=0.2, linewidths=0)
            ax.scatter(g["x_um"], g["y_um"], s=cell_size, c=[color], label=ct, alpha=0.9, linewidths=0)
            
        # Legend for all cell types present in this specific FOV
        present_types = sorted(df_fov["cell type"].dropna().astype(str).unique())
        handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=CELL_COLOR_MAP.get(ct, 'black'), markeredgecolor="none", markersize=8, label=ct) for ct in present_types]

    # --- Grids and Scaling ---
    x_max, y_max = df_fov["x_um"].max(), df_fov["y_um"].max()
    grid_spacing = 25.0 # 50 um grid
    ax.set_axisbelow(True)
    ax.xaxis.set_major_locator(MultipleLocator(100.0))
    ax.yaxis.set_major_locator(MultipleLocator(100.0))

    ax.xaxis.set_minor_locator(MultipleLocator(grid_spacing))
    ax.yaxis.set_minor_locator(MultipleLocator(grid_spacing))
    ax.grid(which="minor", color="white", linestyle="-", linewidth=0.8, alpha=0.7)
    ax.grid(which="major", color="lightgray", linestyle="-", linewidth=0.8, alpha=0.8)

    
    if not pd.isna(x_max) and not pd.isna(y_max):
        add_scale_bar_50um(ax, x_max=x_max, y_max=y_max)

    ax.set_title(title, color="black", fontsize=14)
    ax.set_xlabel("x (µm)", color="black")
    ax.set_ylabel("y (µm)", color="black")
    ax.set_aspect("equal", adjustable="box")
    ax.invert_yaxis()
    ax.tick_params(axis="both", colors="black", labelsize=9)
    for spine in ax.spines.values(): spine.set_visible(False)

    ax.legend(handles=handles, title="Cell type", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9, title_fontsize=10, frameon=True)
    plt.tight_layout()
    plt.show()


In [ ]:
def print_fov_metadata(fov_id):
    """Prints metadata and (optionally) cell counts for a specific FOV."""
    fov_metadata = df_fovs[df_fovs['FOV'] == fov_id]
    if fov_metadata.empty:
        print(f"Error: FOV '{fov_id}' not found in metadata.")
        return
        
    meta = fov_metadata.iloc[0].to_dict()
    print("=" * 50)
    print(f"METADATA FOR FOV: {fov_id}")
    print("=" * 50)
    print(f"Cohort             : {meta.get('Cohort', 'Unknown')}")
    print(f"Patient / Biopsy ID: {meta.get('Patient', 'Unknown')}")
    print(f"Organ              : {meta.get('Organ', 'Unknown')}")
    print(f"Pathological Score : {meta.get('Pathological score', 'Unknown')}")
    print(f"Clinical Score     : {meta.get('Clinical score', 'Unknown')}")
    print(f"Size [um]          : {meta.get('Size [um]', 'Unknown')}")
    
    # Print targeted cell counts if requested
    if df_cells is not None:
        df_fov = df_cells[df_cells["fov"] == fov_id]
        total_cells = len(df_fov)
        print("-" * 50)
        print(f"Total Cells in FOV {fov_id}: {total_cells}")
    print("=" * 50)


In [ ]:
for fov in df_fovs['FOV'].unique():
    print_fov_metadata(fov)
    plot_fov(fov, df_cells, df_fovs)